In [ ]:
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

In [ ]:
import matplotlib.pyplot as plt

plt.rc('font', family='NanumBarunGothic')
plt.plot([1, 2, 3], [1, 4, 9])
plt.title('맑은 고딕 테스트')
plt.show()

In [1]:
# 드라이브에서 파일 가져오기
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import numpy as np

# 파일 경로
img_file_path = '/content/drive/My Drive/apple_images.npy'
label_file_path = '/content/drive/My Drive/apple_labels.npy'

# 파일 로드
img_data = np.load(img_file_path)
label_data = np.load(label_file_path)

print(img_data.shape)
print(label_data.shape)

(1564, 224, 224, 3)
(1564,)


In [3]:
# 0: 특, 1: 상, 2: 보통
# 원본 레이블: 0 => 원 핫 인코딩: [1. 0. 0.]
# 원본 레이블: 1 => 원 핫 인코딩: [0. 1. 0.]
# 원본 레이블: 2 => 원 핫 인코딩: [0. 0. 1.]
from typing import Counter
count = Counter(label_data)
print(count)

Counter({2: 522, 0: 521, 1: 521})


In [4]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical

# 이미지 데이터 정규화 및 타입 변환
img_data = np.array(img_data, dtype=np.float32)
label_data = np.array(label_data, dtype=int)

# 데이터 나누기
x_train, x_test, y_train, y_test = train_test_split(img_data, label_data, test_size=0.1, stratify=label_data, random_state=42)

# One-Hot Encoding
y_train_categorical = to_categorical(y_train, num_classes=3)
y_test_categorical = to_categorical(y_test, num_classes=3)

# CNN 모델 설계
# 다음은 모델에 필요한 모듈을 불러온 후, 모델 층을 쌓는 코드 입니다.
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.models import Sequential

model = Sequential([
    # 입력: 224x224x3
    Conv2D(16, (3, 3), padding='same', activation='relu', input_shape=(224, 224, 3)),
    MaxPooling2D((2, 2)),  # 112x112x16

    Conv2D(32, (3, 3), padding='same', activation='relu'),
    MaxPooling2D((2, 2)),  # 56x56x32

    Conv2D(64, (3, 3), padding='same', activation='relu'),
    MaxPooling2D((2, 2)),  # 28x28x64

    # 이 시점에서 특성 맵 크기 크게 감소
    Conv2D(64, (3, 3), padding='same', activation='relu'),
    GlobalAveragePooling2D(),  # 파라미터 수 크게 감소

    Dense(32, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                      │ (None, 224, 224, 16)        │             448 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d (MaxPooling2D)         │ (None, 112, 112, 16)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_1 (Conv2D)                    │ (None, 112, 112, 32)        │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_1 (MaxPooling2D)       │ (None, 56, 56, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_2 (Conv2D)                    │ (None, 56, 56, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_2 (MaxPooling2D)       │ (None, 28, 28, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 28, 28, 64)          │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 64)                  │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │              99 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 62,691 (244.89 KB)

 Trainable params: 62,691 (244.89 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.metrics import Recall, Precision, AUC

optimizer = Adam(learning_rate=0.001)

# 다중 클래스 분류를 위한 컴파일 설정
model.compile(
   optimizer=optimizer,
   loss='categorical_crossentropy',  # 손실함수
   metrics=['accuracy', 'AUC', 'Precision', 'Recall']
)

# ModelCheckpoint 콜백 설정
model_checkpoint = ModelCheckpoint('best_model.keras', monitor='val_auc', save_best_only=True, mode='max', verbose=1)

In [8]:
# 모델 학습
base_history = model.fit(x_train, y_train_categorical, batch_size=20, validation_split=0.2, epochs=30, callbacks=[model_checkpoint])

Epoch 1/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 8s 76ms/step - AUC: 0.8781 - Precision: 0.7123 - Recall: 0.5783 - accuracy: 0.6823 - loss: 0.5828 - val_AUC: 0.8218 - val_Precision: 0.6038 - val_Recall: 0.5674 - val_accuracy: 0.5922 - val_loss: 0.7157
Epoch 2/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - AUC: 0.8622 - Precision: 0.6930 - Recall: 0.5103 - accuracy: 0.6331 - loss: 0.6060 - val_AUC: 0.8758 - val_Precision: 0.6853 - val_Recall: 0.6099 - val_accuracy: 0.6596 - val_loss: 0.5545
Epoch 3/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - AUC: 0.8684 - Precision: 0.7067 - Recall: 0.5124 - accuracy: 0.6503 - loss: 0.6168 - val_AUC: 0.8758 - val_Precision: 0.6787 - val_Recall: 0.5993 - val_accuracy: 0.6596 - val_loss: 0.5587
Epoch 4/30
57/57 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - AUC: 0.8948 - Precision: 0.7657 - Recall: 0.5670 - accuracy: 0.7126 - loss: 0.5490 - val_AUC: 0.8995 - val_Precision: 0.7419 - val_Recall: 0.6525 - val_accuracy: 0.7163 - val_loss: 0.5193
Epoch 5/30
57/57 ━━━━━━━━━━━━━━━━━━━

In [9]:
# 모델 평가 (여러 지표가 반환되는 경우)
eval_results = model.evaluate(x_test, y_test_categorical)

# 각 평가 지표 출력
print(f"Test Loss: {eval_results[0]}")
print(f"Test Accuracy: {eval_results[1]}")
print(f"AUC: {eval_results[2]}")
print(f"Precision: {eval_results[3]}")
print(f"Recall: {eval_results[4]}")

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - AUC: 0.9903 - Precision: 0.9383 - Recall: 0.9139 - accuracy: 0.9191 - loss: 0.2133
Test Loss: 0.2340647280216217
Test Accuracy: 0.9108280539512634
AUC: 0.9878898859024048
Precision: 0.9220778942108154
Recall: 0.9044585824012756


In [10]:
from sklearn.metrics import accuracy_score

# 모델 예측
y_pred = model.predict(x_test)
y_pred_arg = np.argmax(y_pred, axis=1)

accuracy_score(y_pred_arg, y_test)

5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 97ms/step


0.910828025477707

In [12]:
model.save('/content/drive/MyDrive/apple_dl_model_v1.keras')